# TARDIS — Data Cleaning

This notebook covers the **cleaning** and **preparation** of the SNCF dataset.

Outputs:
- `cleaned_dataset.csv` — ready for downstream analysis
- `cleaning_report.csv` — structured trace of every transformation applied

> All dataset-specific parameters are centralised in the **Configuration** cell (section 1.2).
> To adapt this pipeline to a different dataset, only that cell needs to be modified.

### 1.1 Library imports

In [210]:
import pandas as pd

### 1.2 Configuration

All dataset-specific parameters are defined here.
**This is the only cell that needs to be modified to adapt the pipeline to a different dataset.**

In [211]:
# ── File paths ────────────────────────────────────────────────────────────
INPUT_PATH          = "../data/dataset.csv"
INPUT_SEP           = ";"
OUTPUT_DATASET_PATH = "../cleaned_dataset.csv"
OUTPUT_REPORT_PATH  = "../cleaning_report.csv"

# ── Column: date (parsed to datetime) ─────────────────────────────────────
COL_DATE        = "Date"

# ── Columns: text identifiers ─────────────────────────────────────────────
COL_SERVICE     = "Service"
COL_DEP         = "Departure station"
COL_ARR         = "Arrival station"

# ── Column: base denominator for all rate computations ────────────────────
COL_SCHEDULED   = "Number of scheduled trains"

# ── Column groups ─────────────────────────────────────────────────────────
TEXT_COLS     = [COL_DATE, COL_SERVICE, COL_DEP, COL_ARR]          # excluded from numeric conversion
STR_COLS      = [COL_SERVICE, COL_DEP, COL_ARR]                    # cast to pandas string dtype
LABEL_COLS    = [COL_DEP, COL_ARR, COL_SERVICE]                    # strip + uppercase
CRITICAL_COLS = [COL_DATE, COL_SERVICE, COL_DEP, COL_ARR, COL_SCHEDULED]  # row dropped if null
ROUTE_COLS    = [COL_DEP, COL_ARR]                                 # grouping key for route median

# ── Count columns (cannot be negative) ────────────────────────────────────
COUNT_COLS = [
    "Number of scheduled trains",
    "Number of cancelled trains",
    "Number of trains delayed at departure",
    "Number of trains delayed at arrival",
    "Number of trains delayed > 15min",
    "Number of trains delayed > 30min",
    "Number of trains delayed > 60min",
]

# ── Counts that cannot exceed scheduled trains ────────────────────────────
COUNTS_VS_SCHEDULED = [
    "Number of cancelled trains",
    "Number of trains delayed at departure",
    "Number of trains delayed at arrival",
]

# ── Delay hierarchy: each col must be <= the previous one ─────────────────
# Listed in ascending threshold order (>15 >= >30 >= >60)
DELAY_HIERARCHY = [
    "Number of trains delayed > 15min",
    "Number of trains delayed > 30min",
    "Number of trains delayed > 60min",
]

# ── Derivable delay columns: departure ───────────────────────────────────
COL_DEP_LATE  = "Average delay of late trains at departure"
COL_DEP_ALL   = "Average delay of all trains at departure"
COL_N_DEP     = "Number of trains delayed at departure"

# ── Derivable delay columns: arrival ────────────────────────────────────
COL_ARR_LATE  = "Average delay of late trains at arrival"
COL_ARR_ALL   = "Average delay of all trains at arrival"
COL_N_ARR     = "Number of trains delayed at arrival"

# ── Rate source columns ──────────────────────────────────────────────────
COL_CANCELLED  = "Number of cancelled trains"

# ── Pattern: columns to drop ─────────────────────────────────────────────
COMMENTS_REGEX = "comments$"

# ── Feature: delay category ──────────────────────────────────────────────
COL_DELAY_SOURCE = COL_ARR_ALL
DELAY_BINS       = [float('-inf'), 0, 5, 15, 30, float('inf')]
DELAY_LABELS     = ['early', 'on_time', 'slight', 'moderate', 'severe']

# ── Feature: new column names ────────────────────────────────────────────
COL_YEAR        = "year"
COL_MONTH       = "month"
COL_SEASON      = "season"
COL_DELAY_CAT   = "delay_category"
COL_CANCEL_RATE = "cancellation_rate"
COL_PUNCT_RATE  = "punctuality_rate"

print('Configuration loaded.')

Configuration loaded.


> *Tracking — initialising report accumulator*

In [212]:
report = []

### 2. Data loading

We load the dataset using the parameters defined in the configuration and keep a raw copy.

In [213]:
df = pd.read_csv(INPUT_PATH, sep=INPUT_SEP)
original_file = df.copy()
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')

Loaded: 12070 rows, 26 columns


> *Tracking — initial dataset state*

In [214]:
_initial_null_total = original_file.isnull().sum().sum()
_perfect_rows       = original_file.dropna().shape[0]
_perfect_cols       = original_file.dropna(axis=1).shape[1]

report += [
    {'metric': 'initial_rows',               'value': original_file.shape[0],                 'category': 'initial_state', 'reason': 'Raw dataset before any cleaning'},
    {'metric': 'initial_columns',            'value': original_file.shape[1],                 'category': 'initial_state', 'reason': 'Raw dataset before any cleaning'},
    {'metric': 'initial_null_total',         'value': int(_initial_null_total),               'category': 'initial_state', 'reason': 'Total missing values across all cells'},
    {'metric': 'perfect_rows',               'value': _perfect_rows,                          'category': 'initial_state', 'reason': 'Rows with zero NaN — no cleaning needed'},
    {'metric': 'rows_lost_if_strict_dropna', 'value': original_file.shape[0] - _perfect_rows, 'category': 'initial_state', 'reason': 'Rows lost with a strict zero-NaN policy'},
    {'metric': 'perfect_columns',            'value': _perfect_cols,                          'category': 'initial_state', 'reason': 'Columns with zero NaN'},
    {'metric': 'cols_lost_if_strict_dropna', 'value': original_file.shape[1] - _perfect_cols, 'category': 'initial_state', 'reason': 'Columns lost with a strict zero-NaN policy'},
]
for col in original_file.columns:
    report.append({'metric': f'initial_null__{col}', 'value': int(original_file[col].isnull().sum()), 'category': 'initial_null_per_column', 'reason': col})

print(f'Initial null total : {_initial_null_total}')
print(f'Perfect rows       : {_perfect_rows} / {original_file.shape[0]}')
print(f'Perfect columns    : {_perfect_cols} / {original_file.shape[1]}')

Initial null total : 38874
Perfect rows       : 2 / 12070
Perfect columns    : 0 / 26


### 2.1 Removing comment columns

Columns matching `{COMMENTS_REGEX}` contain free text — not usable in a tabular framework.

In [215]:
comment_cols = df.filter(regex=COMMENTS_REGEX).columns.tolist()
df = df.drop(columns=comment_cols)
print(f'Dropped {len(comment_cols)} column(s): {comment_cols}')
print(f'Remaining: {df.shape[1]} columns')

Dropped 3 column(s): ['Cancellation comments', 'Departure delay comments', 'Arrival delay comments']
Remaining: 23 columns


> *Tracking — columns removed*

In [216]:
for col in comment_cols:
    report.append({'metric': 'column_dropped', 'value': col, 'category': 'cleaning', 'reason': 'Irrelevant data — free text comment column'})
report.append({'metric': 'columns_dropped_total', 'value': len(comment_cols), 'category': 'cleaning', 'reason': 'Irrelevant data — free text columns removed'})
print(f'Tracked: {len(comment_cols)} column(s) logged.')

Tracked: 3 column(s) logged.


### 3. Duplicate check

Duplicate rows bias statistics by counting the same observation more than once.

In [217]:
nb_dup = df.duplicated().sum()
print(f'Duplicates found: {nb_dup}')
if nb_dup > 0:
    df = df.drop_duplicates()
    print(f'Removed. Rows remaining: {len(df)}')
else:
    print('No duplicates found.')

Duplicates found: 174
Removed. Rows remaining: 11896


> *Tracking — rows removed — duplicates*

In [218]:
report.append({'metric': 'rows_dropped_duplicates', 'value': int(nb_dup), 'category': 'cleaning', 'reason': 'Duplicate rows — same observation counted more than once'})
print(f'Tracked: {nb_dup} row(s) logged.')

Tracked: 174 row(s) logged.


### 4. Missing values — overview

Full picture of missing values per column before any removal or imputation.

In [219]:
nan_counts = df.isnull().sum()
print('Missing values per column:')
print(nan_counts[nan_counts > 0].to_string())
print(f'\nTotal: {nan_counts.sum()}')

Missing values per column:
Date                                                                              59
Service                                                                          240
Departure station                                                                 59
Arrival station                                                                   59
Average journey time                                                             236
Number of scheduled trains                                                       237
Number of cancelled trains                                                       237
Number of trains delayed at departure                                            238
Average delay of late trains at departure                                        238
Average delay of all trains at departure                                         237
Number of trains delayed at arrival                                              237
Average delay of late trains at arriva

### 4.1 Dropping rows with missing critical data

We drop rows only where a column defined in `CRITICAL_COLS` is null.
These columns are essential and cannot be derived from other row data.
All other NaN values are handled later (steps 4.6 and 4.7).

In [220]:
before = len(df)
df = df.dropna(subset=CRITICAL_COLS)
print(f'Critical columns : {CRITICAL_COLS}')
print(f'Rows dropped     : {before - len(df)}')
print(f'Rows remaining   : {len(df)}')

Critical columns : ['Date', 'Service', 'Departure station', 'Arrival station', 'Number of scheduled trains']
Rows dropped     : 636
Rows remaining   : 11260


> *Tracking — rows removed — critical NaN*

In [221]:
_crit_dropped = before - len(df)
report.append({'metric': 'rows_dropped_critical_nan', 'value': _crit_dropped, 'category': 'cleaning', 'reason': 'Unusable for analysis — missing key identifiers (CRITICAL_COLS)'})
print(f'Tracked: {_crit_dropped} row(s) logged.')

Tracked: 636 row(s) logged.


### 4.2 Date standardization

The date column is parsed into `datetime64`. `format='mixed'` handles heterogeneous strings.

In [222]:
df[COL_DATE] = pd.to_datetime(df[COL_DATE], format='mixed')
print(f'dtype : {df[COL_DATE].dtype}')
print(f'Range : {df[COL_DATE].min().date()} → {df[COL_DATE].max().date()}')

dtype : datetime64[ns]
Range : 2018-01-01 → 2025-12-01


### 4.3 Numeric column conversion

Columns not in `TEXT_COLS` are converted to numeric.
We guard with a dtype check first to avoid silently producing NaN via `.str` on already-numeric columns.

In [223]:
numeric_cols = [c for c in df.columns if c not in TEXT_COLS]

for col in numeric_cols:
    if df[col].dtype == object:
        df[col] = df[col].str.strip().str.replace(',', '.')
    df[col] = pd.to_numeric(df[col], errors='coerce')

still_object = [c for c in numeric_cols if df[c].dtype == object]
if still_object:
    print(f'WARNING — still object after conversion: {still_object}')
else:
    print(f'All {len(numeric_cols)} numeric columns successfully converted.')

All 19 numeric columns successfully converted.


### 4.4 Text column dtype

Columns in `STR_COLS` are explicitly cast to pandas `string` dtype to prevent mixed-type issues.

In [224]:
df[STR_COLS] = df[STR_COLS].astype('string')
print('Dtypes after cast:')
print(df[STR_COLS].dtypes.to_string())

Dtypes after cast:
Service              string[python]
Departure station    string[python]
Arrival station      string[python]


### 4.5 Label normalization

Columns in `LABEL_COLS` are stripped and uppercased to eliminate formatting duplicates.

In [225]:
for col in LABEL_COLS:
    df[col] = df[col].str.strip().str.upper()

for col in LABEL_COLS:
    print(f'Unique {col}: {df[col].nunique()}')

Unique Departure station: 64
Unique Arrival station: 63
Unique Service: 2


### 4.6 Filling departure delay NaN from row data

The two departure delay columns are mathematically linked:
- `COL_DEP_LATE = COL_DEP_ALL × (COL_SCHEDULED / COL_N_DEP)`
- `COL_DEP_ALL  = COL_DEP_LATE × (COL_N_DEP / COL_SCHEDULED)`

In [226]:
mask = df[COL_DEP_LATE].isna() & df[COL_DEP_ALL].notna() & (df[COL_N_DEP] > 0)
_dep_late_filled = int(mask.sum())
df.loc[mask, COL_DEP_LATE] = df.loc[mask, COL_DEP_ALL] * df.loc[mask, COL_SCHEDULED] / df.loc[mask, COL_N_DEP]
print(f'COL_DEP_LATE filled from COL_DEP_ALL : {_dep_late_filled} row(s)')

mask = df[COL_DEP_ALL].isna() & df[COL_DEP_LATE].notna() & df[COL_N_DEP].notna()
_dep_all_filled = int(mask.sum())
df.loc[mask, COL_DEP_ALL] = df.loc[mask, COL_DEP_LATE] * df.loc[mask, COL_N_DEP] / df.loc[mask, COL_SCHEDULED]
print(f'COL_DEP_ALL  filled from COL_DEP_LATE : {_dep_all_filled} row(s)')

COL_DEP_LATE filled from COL_DEP_ALL : 269 row(s)
COL_DEP_ALL  filled from COL_DEP_LATE : 279 row(s)


> *Tracking — departure delay NaN filled*

In [227]:
report += [
    {'metric': 'values_filled_dep_late', 'value': _dep_late_filled, 'category': 'data_recovery', 'reason': 'Computed from average departure delay + scheduled/delayed counts'},
    {'metric': 'values_filled_dep_all',  'value': _dep_all_filled,  'category': 'data_recovery', 'reason': 'Computed from late-train departure delay + delayed/scheduled counts'},
]
print(f'Tracked: dep_late={_dep_late_filled}, dep_all={_dep_all_filled}')

Tracked: dep_late=269, dep_all=279


### 4.7 Filling arrival delay NaN from row data

Same derivation logic applied to `COL_ARR_LATE` and `COL_ARR_ALL`.

In [228]:
mask = df[COL_ARR_LATE].isna() & df[COL_ARR_ALL].notna() & (df[COL_N_ARR] > 0)
_arr_late_filled = int(mask.sum())
df.loc[mask, COL_ARR_LATE] = df.loc[mask, COL_ARR_ALL] * df.loc[mask, COL_SCHEDULED] / df.loc[mask, COL_N_ARR]
print(f'COL_ARR_LATE filled from COL_ARR_ALL : {_arr_late_filled} row(s)')

mask = df[COL_ARR_ALL].isna() & df[COL_ARR_LATE].notna() & df[COL_N_ARR].notna()
_arr_all_filled = int(mask.sum())
df.loc[mask, COL_ARR_ALL] = df.loc[mask, COL_ARR_LATE] * df.loc[mask, COL_N_ARR] / df.loc[mask, COL_SCHEDULED]
print(f'COL_ARR_ALL  filled from COL_ARR_LATE : {_arr_all_filled} row(s)')

COL_ARR_LATE filled from COL_ARR_ALL : 266 row(s)
COL_ARR_ALL  filled from COL_ARR_LATE : 266 row(s)


> *Tracking — arrival delay NaN filled*

In [229]:
report += [
    {'metric': 'values_filled_arr_late', 'value': _arr_late_filled, 'category': 'data_recovery', 'reason': 'Computed from average arrival delay + scheduled/delayed counts'},
    {'metric': 'values_filled_arr_all',  'value': _arr_all_filled,  'category': 'data_recovery', 'reason': 'Computed from late-train arrival delay + delayed/scheduled counts'},
]
print(f'Tracked: arr_late={_arr_late_filled}, arr_all={_arr_all_filled}')

Tracked: arr_late=266, arr_all=266


### 4.8 Remaining NaN — final report

Any NaN left at this point is intentionally kept as null — these columns cannot be derived from row data.

In [230]:
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]
if not remaining.empty:
    print('Columns with remaining NaN (left as null intentionally):')
    print(remaining.to_string())
else:
    print('No remaining NaN.')

Columns with remaining NaN (left as null intentionally):
Average journey time                                                             286
Number of cancelled trains                                                       223
Number of trains delayed at departure                                            228
Average delay of late trains at departure                                         10
Average delay of all trains at departure                                          10
Number of trains delayed at arrival                                              230
Average delay of late trains at arrival                                           18
Average delay of all trains at arrival                                             9
Number of trains delayed > 15min                                                 222
Average delay of trains > 15min (if competing with flights)                      273
Number of trains delayed > 30min                                                 223
Number o

### 5. Cleaning summary

Shape comparison between raw and cleaned dataset.

In [231]:
print(f'Original : {original_file.shape[0]} rows, {original_file.shape[1]} columns')
print(f'Cleaned  : {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Removed  : {original_file.shape[0] - df.shape[0]} rows, {original_file.shape[1] - df.shape[1]} columns')

Original : 12070 rows, 26 columns
Cleaned  : 11260 rows, 23 columns
Removed  : 810 rows, 3 columns


### 6. Year and month extraction

In [232]:
df[COL_YEAR]  = df[COL_DATE].dt.year
df[COL_MONTH] = df[COL_DATE].dt.month
print(f'Years  : {sorted(df[COL_YEAR].unique().tolist())}')
print(f'Months : {sorted(df[COL_MONTH].unique().tolist())}')

Years  : [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Months : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


### 7. Season encoding

In [233]:
season_map = {
    12:'winter',1:'winter',2:'winter',
    3:'spring',4:'spring',5:'spring',
    6:'summer',7:'summer',8:'summer',
    9:'autumn',10:'autumn',11:'autumn',
}
df[COL_SEASON] = df[COL_MONTH].map(season_map)
print('Season distribution:')
print(df[COL_SEASON].value_counts().to_string())

Season distribution:
season
summer    2832
autumn    2820
winter    2818
spring    2790


### 8. Delay severity categorization

Bins and labels are defined in `DELAY_BINS` / `DELAY_LABELS` in the configuration.

In [234]:
df[COL_DELAY_CAT] = pd.cut(df[COL_DELAY_SOURCE], bins=DELAY_BINS, labels=DELAY_LABELS)
print('Delay category distribution:')
print(df[COL_DELAY_CAT].value_counts().sort_index().to_string())

Delay category distribution:
delay_category
early        219
on_time     4951
slight      5693
moderate     373
severe        15


### 9. Cancellation rate

In [235]:
df[COL_CANCEL_RATE] = df[COL_CANCELLED] / df[COL_SCHEDULED]
print(f'{COL_CANCEL_RATE} — min: {df[COL_CANCEL_RATE].min():.4f}  mean: {df[COL_CANCEL_RATE].mean():.4f}  max: {df[COL_CANCEL_RATE].max():.4f}')

cancellation_rate — min: 0.0000  mean: inf  max: inf


### 10. Punctuality rate

In [236]:
df[COL_PUNCT_RATE] = 1 - (df[COL_N_ARR] / df[COL_SCHEDULED])
print(f'{COL_PUNCT_RATE} — min: {df[COL_PUNCT_RATE].min():.4f}  mean: {df[COL_PUNCT_RATE].mean():.4f}  max: {df[COL_PUNCT_RATE].max():.4f}')

punctuality_rate — min: 0.0000  mean: 0.8610  max: 1.0000


### 11. Negative value correction

All columns in `COUNT_COLS` must be non-negative.
Violations are replaced by the route-level median (`ROUTE_COLS`).

In [237]:
_neg_fixed_total = 0
for col in COUNT_COLS:
    mask = df[col] < 0
    if mask.any():
        route_median = df.groupby(ROUTE_COLS)[col].transform(lambda x: x[x >= 0].median())
        df.loc[mask, col] = route_median[mask]
        _neg_fixed_total += int(mask.sum())
        print(f'[FIXED] {col}: {mask.sum()} negative value(s) → route median')
print(f'Total: {_neg_fixed_total} fix(es)')

[FIXED] Number of trains delayed > 30min: 41 negative value(s) → route median
Total: 41 fix(es)


> *Tracking — negative value corrections*

In [238]:
report.append({'metric': 'values_fixed_negative', 'value': _neg_fixed_total, 'category': 'data_correction', 'reason': 'Impossible negative count values — replaced by route-level median'})
print(f'Tracked: {_neg_fixed_total} fix(es) logged.')

Tracked: 41 fix(es) logged.


### 11.1 Consistency: counts cannot exceed scheduled trains

Columns in `COUNTS_VS_SCHEDULED` cannot exceed `COL_SCHEDULED` on the same row.

In [239]:
_cons_count_fixed = 0
for col in COUNTS_VS_SCHEDULED:
    mask = df[col].notna() & (df[col] > df[COL_SCHEDULED])
    if mask.any():
        route_median = df.groupby(ROUTE_COLS)[col].transform(
            lambda x, c=col: x[x <= df.loc[x.index, COL_SCHEDULED]].median()
        )
        df.loc[mask, col] = route_median[mask]
        _cons_count_fixed += int(mask.sum())
        print(f'[FIXED] {col}: {mask.sum()} value(s) exceeded scheduled → route median')
print(f'Total: {_cons_count_fixed} fix(es)')

[FIXED] Number of cancelled trains: 54 value(s) exceeded scheduled → route median
[FIXED] Number of trains delayed at departure: 15 value(s) exceeded scheduled → route median
Total: 69 fix(es)


> *Tracking — consistency — counts vs scheduled*

In [240]:
report.append({'metric': 'values_fixed_count_vs_scheduled', 'value': _cons_count_fixed, 'category': 'data_correction', 'reason': 'Counts exceeding scheduled trains — logically impossible'})
print(f'Tracked: {_cons_count_fixed} fix(es) logged.')

Tracked: 69 fix(es) logged.


### 11.2 Consistency: delay hierarchy

The order defined in `DELAY_HIERARCHY` must hold: each column must be ≤ the previous one.
Violations are fixed bottom-up (highest threshold first).

In [241]:
_hier_fixed = 0
for i in range(len(DELAY_HIERARCHY) - 1):
    col_higher = DELAY_HIERARCHY[i + 1]   # higher threshold → fewer trains
    col_lower  = DELAY_HIERARCHY[i]        # lower threshold → more trains
    mask = df[col_higher].notna() & df[col_lower].notna() & (df[col_higher] > df[col_lower])
    if mask.any():
        route_median = df.groupby(ROUTE_COLS)[col_higher].transform(
            lambda x, cl=col_lower: x[x <= df.loc[x.index, cl]].median()
        )
        df.loc[mask, col_higher] = route_median[mask]
        _hier_fixed += int(mask.sum())
        print(f'[FIXED] {col_higher} > {col_lower}: {mask.sum()} row(s)')
print(f'Total: {_hier_fixed} fix(es)')

[FIXED] Number of trains delayed > 30min > Number of trains delayed > 15min: 3 row(s)
[FIXED] Number of trains delayed > 60min > Number of trains delayed > 30min: 315 row(s)
Total: 318 fix(es)


> *Tracking — consistency — delay hierarchy*

In [242]:
report.append({'metric': 'values_fixed_delay_hierarchy', 'value': _hier_fixed, 'category': 'data_correction', 'reason': 'Delay hierarchy violation — higher threshold exceeded lower threshold'})
print(f'Tracked: {_hier_fixed} fix(es) logged.')

Tracked: 318 fix(es) logged.


### 11.3 Recomputing derived rates after all corrections

We recompute `COL_CANCEL_RATE` and `COL_PUNCT_RATE` from the corrected source columns.

In [243]:
df[COL_CANCEL_RATE] = df[COL_CANCELLED] / df[COL_SCHEDULED]
df[COL_PUNCT_RATE]  = 1 - (df[COL_N_ARR] / df[COL_SCHEDULED])

print(f'{COL_CANCEL_RATE} — min: {df[COL_CANCEL_RATE].min():.4f}  mean: {df[COL_CANCEL_RATE].mean():.4f}  max: {df[COL_CANCEL_RATE].max():.4f}')
print(f'{COL_PUNCT_RATE}  — min: {df[COL_PUNCT_RATE].min():.4f}  mean: {df[COL_PUNCT_RATE].mean():.4f}  max: {df[COL_PUNCT_RATE].max():.4f}')

cancellation_rate — min: 0.0000  mean: inf  max: inf
punctuality_rate  — min: 0.0000  mean: 0.8610  max: 1.0000


### 12. Export cleaned dataset

Reset index and write to `OUTPUT_DATASET_PATH`.

In [244]:
df = df.reset_index(drop=True)
df.to_csv(OUTPUT_DATASET_PATH, index=False)
print(f'Exported: {len(df)} rows, {len(df.columns)} columns → {OUTPUT_DATASET_PATH}')
print(f'Columns: {df.columns.tolist()}')

Exported: 11260 rows, 29 columns → ../cleaned_dataset.csv
Columns: ['Date', 'Service', 'Departure station', 'Arrival station', 'Average journey time', 'Number of scheduled trains', 'Number of cancelled trains', 'Number of trains delayed at departure', 'Average delay of late trains at departure', 'Average delay of all trains at departure', 'Number of trains delayed at arrival', 'Average delay of late trains at arrival', 'Average delay of all trains at arrival', 'Number of trains delayed > 15min', 'Average delay of trains > 15min (if competing with flights)', 'Number of trains delayed > 30min', 'Number of trains delayed > 60min', 'Pct delay due to external causes', 'Pct delay due to infrastructure', 'Pct delay due to traffic management', 'Pct delay due to rolling stock', 'Pct delay due to station management and equipment reuse', 'Pct delay due to passenger handling (crowding, disabled persons, connections)', 'year', 'month', 'season', 'delay_category', 'cancellation_rate', 'punctuality_r

### 13. Cleaning report export

Final state is appended to the report accumulator, then written to `OUTPUT_REPORT_PATH`.

In [245]:
_remaining_null = int(df.isnull().sum().sum())
report += [
    {'metric': 'final_rows',         'value': len(df),                              'category': 'final_state', 'reason': 'Rows after full cleaning pipeline'},
    {'metric': 'final_columns',      'value': len(df.columns),                      'category': 'final_state', 'reason': 'Columns after cleaning + feature engineering'},
    {'metric': 'final_null_total',   'value': _remaining_null,                      'category': 'final_state', 'reason': 'Intentionally kept null — non-derivable columns'},
    {'metric': 'total_rows_removed', 'value': original_file.shape[0] - len(df),     'category': 'final_state', 'reason': 'Total rows removed across all cleaning steps'},
]

report_df = pd.DataFrame(report, columns=['metric', 'value', 'category', 'reason'])
report_df.to_csv(OUTPUT_REPORT_PATH, index=False)

print(f'Report saved: {len(report_df)} entries → {OUTPUT_REPORT_PATH}')
print(report_df['category'].value_counts().to_string())

Report saved: 50 entries → ../cleaning_report.csv
category
initial_null_per_column    26
initial_state               7
cleaning                    6
data_recovery               4
final_state                 4
data_correction             3
